# 분류 모델 — 한우 최종등급 예측 (GPU 버전)
LightGBM(CPU) / XGBoost(GPU) / CatBoost(GPU) / Logistic(전체) → Optuna 튜닝 → LGB+XGB 스태킹 앙상블

In [1]:
# ==============================================================
# 라이브러리 + 한글 폰트
# ==============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform, os, json, warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, accuracy_score,
                             classification_report, confusion_matrix)
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
import shap
import joblib

sns.set_style("whitegrid")
def set_korean_font():
    s = platform.system()
    if s == "Darwin":      plt.rcParams["font.family"] = "AppleGothic"
    elif s == "Windows":   plt.rcParams["font.family"] = "Malgun Gothic"
    else:                  plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
set_korean_font()

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 100)
os.makedirs("../../../figures", exist_ok=True)
os.makedirs("../../../models", exist_ok=True)
os.makedirs("../../../data/processed/4_model", exist_ok=True)
SEED = 42

# GPU 사용 가능 여부 확인
try:
    import subprocess
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                            capture_output=True, text=True, timeout=5)
    GPU_AVAILABLE = result.returncode == 0
    if GPU_AVAILABLE:
        print(f"GPU 감지: {result.stdout.strip()}")
        print("XGBoost → device='cuda', CatBoost → task_type='GPU'")
    else:
        print("GPU 없음 — XGBoost/CatBoost CPU 폴백")
except Exception:
    GPU_AVAILABLE = False
    print("GPU 확인 실패 — CPU로 실행")

XGB_DEVICE  = "cuda" if GPU_AVAILABLE else "cpu"
CAT_TASK    = "GPU"  if GPU_AVAILABLE else "CPU"

GPU 감지: NVIDIA GeForce RTX 4060 Ti
XGBoost → device='cuda', CatBoost → task_type='GPU'


In [2]:
# ==============================================================
# step11 로드 + 분할(split_assignment) 병합
# ==============================================================
df = pd.read_csv("../../../data/step11_features.csv",
                 encoding="utf-8-sig", low_memory=False)
print(f"데이터: {df.shape}")

SPLIT_PATH = "../../../data/split_assignment.csv"

if os.path.exists(SPLIT_PATH):
    split = pd.read_csv(SPLIT_PATH, encoding="utf-8-sig")
    print("기존 분할 로드 (회귀팀과 공유)")
else:
    from sklearn.model_selection import StratifiedGroupKFold
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    dev_idx, hold_idx = next(sgkf.split(df, df["LAST_GRADE"],
                                        groups=df["FARM_UNIQUE_NO"]))
    df["split"] = "dev"; df.loc[hold_idx, "split"] = "holdout"
    df["fold"] = -1
    dev_tmp = df[df["split"] == "dev"]
    sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for k, (_, va) in enumerate(sgkf2.split(dev_tmp, dev_tmp["LAST_GRADE"],
                                            groups=dev_tmp["FARM_UNIQUE_NO"])):
        df.loc[dev_tmp.index[va], "fold"] = k
    df[["CATTLE_NO","split","fold"]].to_csv(SPLIT_PATH, index=False, encoding="utf-8-sig")
    split = df[["CATTLE_NO","split","fold"]]
    print("분할 새로 생성 후 저장")

df = df.merge(split, on="CATTLE_NO", how="left") if "split" not in df.columns else df
print(df["split"].value_counts())

farm_check = df.groupby("FARM_UNIQUE_NO")["split"].nunique()
assert (farm_check > 1).sum() == 0, "농장 누수 발견!"
print("농장 누수 검증 통과")

데이터: (2408699, 104)
기존 분할 로드 (회귀팀과 공유)
split
dev        1926959
holdout     481740
Name: count, dtype: int64
농장 누수 검증 통과


In [3]:
# ==============================================================
# 트랙1 피처 구성 (누수 차단)
# ==============================================================
EXCLUDE = {
    "LAST_GRADE", "grade_num",
    "BACKFAT","REA","WINDEX","INSFAT","YUKSAK","FATSAK","TISSUE","GROWTH",
    "WGRADE","COST_AMT",
    "CATTLE_NO","FARM_UNIQUE_NO","stn",
    "KPN_NO","FATHER_CATTLE_NO","MOTHER_ANIMAL_NO",
    "F_GMOTHER_ANIMAL_NO","F_GFATHER_CATTLE_NO",
    "M_GMOTHER_ANIMAL_NO","M_GFATHER_CATTLE_NO",
    "ABATT_DATE","JUDGE_DATE","BIRTH_YMD",
    "sido","sigungu","eupmyeondong","JUDGE_SEX",
    "abatt_season","birth_season","farm_size","고온_bin",
    "days_폐사",
    "split","fold",
}
features_track1 = [c for c in df.columns
                   if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]
print(f"트랙1 피처: {len(features_track1)}개")

QUALITY = ["BACKFAT","REA","WINDEX","INSFAT","YUKSAK","FATSAK","TISSUE","GROWTH"]
features_post = features_track1 + QUALITY
print(f"참고-사후 피처: {len(features_post)}개")

트랙1 피처: 70개
참고-사후 피처: 78개


In [4]:
# ==============================================================
# 등급 → 정수 (16개)
# ==============================================================
GRADE_ORDER = ["등외","3C","3B","3A","2C","2B","2A","1C","1B","1A",
               "1+C","1+B","1+A","1++C","1++B","1++A"]
grade_to_idx = {g: i for i, g in enumerate(GRADE_ORDER)}
idx_to_grade = {i: g for g, i in grade_to_idx.items()}

df["y"] = df["LAST_GRADE"].map(grade_to_idx)
assert df["y"].isnull().sum() == 0, "매핑 안 된 등급 존재!"
n_classes = 16
print(df["LAST_GRADE"].value_counts().reindex(GRADE_ORDER).to_string())

LAST_GRADE
등외        5540
3C       23542
3B       97576
3A       47214
2C       62938
2B      195866
2A      127410
1C      116976
1B      299290
1A      168174
1+C     130094
1+B     311222
1+A     167246
1++C    128997
1++B    319588
1++A    207026


In [5]:
# ==============================================================
# dev / holdout 분리 + 결측 처리 유틸
# ==============================================================
dev  = df[df["split"] == "dev"].reset_index(drop=True)
hold = df[df["split"] == "holdout"].reset_index(drop=True)
print(f"개발: {len(dev):,} / 홀드아웃: {len(hold):,}")

def prepare_X(train_X, *other_Xs):
    train_X = train_X.replace([np.inf, -np.inf], np.nan)
    med = train_X.median(numeric_only=True)
    out = [train_X.fillna(med)]
    for X_ in other_Xs:
        out.append(X_.replace([np.inf, -np.inf], np.nan).fillna(med))
    bad = out[0].columns[out[0].isnull().any()].tolist()
    if bad:
        print(f"  [방어] 전부 결측 컬럼 제거: {bad}")
        out = [o.drop(columns=bad) for o in out]
    return out if len(out) > 1 else out[0]

fold_indices = [(np.where(dev["fold"] != k)[0], np.where(dev["fold"] == k)[0])
                for k in range(5)]
y_dev = dev["y"]

HEAT_Q75 = float(np.nanquantile(dev["ratio_고온"], 0.75))
for part in (dev, hold):
    part["heat_high"] = (part["ratio_고온"] >= HEAT_Q75).astype(int)
X_t1 = dev[features_track1]
print(f"heat_high 임계값(dev 75분위) = {HEAT_Q75:.4f}")

개발: 1,926,959 / 홀드아웃: 481,740
heat_high 임계값(dev 75분위) = 0.4486


## 파생변수 추가·검증 하니스
격리 CV(2-fold)로 하나씩 검증 → 채택분 + OOF 타깃 인코딩(KPN_NO / FATHER / FARM)을 features_track1에 반영

In [ ]:
# ============================================================
# 파생변수 추가·검증 하니스 — 하나씩 격리 CV (오르면 채택/내리면 기각)
# ============================================================
from sklearn.model_selection import train_test_split

SMOOTHING_KPN  = 20   # 혈통 스무딩
SMOOTHING_FARM = 50   # 농장 스무딩 (환경효과 섞임 → 더 보수적)

def add_candidates(d):
    """타깃 미사용 파생변수만. fold 구분 불필요."""
    cands = {}

    # 1. daily_gain 비선형
    d["daily_gain_squared"] = d["daily_gain"] ** 2
    cands["daily_gain_squared"] = ["daily_gain_squared"]

    q75 = d["daily_gain"].quantile(0.75)
    d["gain_high"] = (d["daily_gain"] >= q75).astype(int)
    cands["gain_high"] = ["gain_high"]

    # 2. 월령(AGE) 또래 대비 체격
    d["weight_vs_age_peer"] = d["WEIGHT"] / d.groupby("AGE")["WEIGHT"].transform("mean")
    cands["weight_vs_age_peer"] = ["weight_vs_age_peer"]

    # 3. 혈통 미등록 여부
    d["lineage_missing"] = d["KPN_NO_freq"].isnull().astype(int)
    cands["lineage_missing"] = ["lineage_missing"]

    # ↓↓↓ 추가 파생변수는 여기에 ↓↓↓
    return cands

# OOF 타깃 인코딩 후보
TE_CANDIDATES = {
    "KPN_NO_te":           ("KPN_NO",           SMOOTHING_KPN),
    "FATHER_CATTLE_NO_te": ("FATHER_CATTLE_NO",  SMOOTHING_KPN),
    "FARM_UNIQUE_NO_te":   ("FARM_UNIQUE_NO",    SMOOTHING_FARM),
}

cands = add_candidates(dev)

PARAMS_ISO = dict(objective="multiclass", num_class=n_classes,
                  learning_rate=0.05, n_estimators=500,
                  max_depth=8, num_leaves=101, min_child_samples=66,
                  subsample=0.888, subsample_freq=1, colsample_bytree=0.813,
                  class_weight="balanced", n_jobs=-1, verbose=-1, random_state=SEED)

def _add_te_fold(tr, va, col, enc_name, smoothing):
    global_mean = tr["y"].mean()
    stats = tr.groupby(col)["y"].agg(["mean", "count"])
    stats[enc_name] = (stats["mean"]*stats["count"] + global_mean*smoothing) / (stats["count"]+smoothing)
    enc_map = stats[enc_name].to_dict()
    tr = tr.copy(); va = va.copy()
    tr[enc_name] = tr[col].map(enc_map).fillna(global_mean)
    va[enc_name] = va[col].map(enc_map).fillna(global_mean)
    return tr, va

def iso_cv(extra, n_folds=2, sub=200_000, te_name=None):
    sc = []
    for k in range(n_folds):
        tr = dev[dev["fold"] != k].copy()
        va = dev[dev["fold"] == k].copy()
        trs, _ = train_test_split(tr, train_size=min(sub, len(tr)),
                                  stratify=tr["y"], random_state=SEED)
        extra_new = [c for c in extra if c not in features_track1]
        cols = features_track1 + extra_new
        if te_name:
            col, smoothing = TE_CANDIDATES[te_name]
            trs, va = _add_te_fold(trs, va, col, te_name, smoothing)
            if te_name not in cols:
                cols = cols + [te_name]
        m = lgb.LGBMClassifier(**PARAMS_ISO).fit(
            trs[cols], trs["y"], eval_set=[(va[cols], va["y"])],
            callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)])
        sc.append(f1_score(va["y"], np.asarray(m.predict(va[cols])).ravel(), average="macro"))
    return np.mean(sc)

# 검증 실행
base = iso_cv([])
print(f"기준(현재 {len(features_track1)}피처) = {base:.4f}\n")

adopted = []

for name, cols in cands.items():
    s = iso_cv(cols)
    if s - base > 0.0008:
        verdict = "✅ 채택"; adopted.extend(cols)
    elif abs(s - base) <= 0.0008:
        verdict = "⚪ 노이즈/무효"
    else:
        verdict = "❌ 기각"
    print(f"  +{name:28s} {s:.4f}  ({s-base:+.4f})  {verdict}")

print()

for te_name in TE_CANDIDATES:
    s = iso_cv([], te_name=te_name)
    if s - base > 0.0008:
        verdict = "✅ 채택"; adopted.append(te_name)
    elif abs(s - base) <= 0.0008:
        verdict = "⚪ 노이즈/무효"
    else:
        verdict = "❌ 기각"
    print(f"  +{te_name:28s} {s:.4f}  ({s-base:+.4f})  {verdict}  [OOF TE]")

# 단순 파생변수를 features_track1에 반영
for col in adopted:
    if col not in features_track1 and col not in TE_CANDIDATES:
        features_track1.append(col)

print(f"\n채택된 파생변수: {adopted if adopted else '없음'}")
print(f"갱신된 features_track1: {len(features_track1)}개")
print("※ TE 변수는 아래 셀에서 OOF 방식으로 추가")

기준(현재 70피처) = 0.2056



In [ ]:
# ==============================================================
# X_t1 확정 — 채택된 TE 변수 OOF 방식으로 dev·hold 전체에 추가
# ==============================================================
adopted_te = [te for te in TE_CANDIDATES if te in adopted]
if adopted_te:
    print("OOF 타깃 인코딩 적용 중...")
    for te_name in adopted_te:
        col, smoothing = TE_CANDIDATES[te_name]
        global_mean = dev["y"].mean()
        dev[te_name] = np.nan
        for k in range(5):
            tr_idx_k = dev["fold"] != k
            va_idx_k = dev["fold"] == k
            stats = dev[tr_idx_k].groupby(col)["y"].agg(["mean", "count"])
            stats[te_name] = (stats["mean"]*stats["count"] + global_mean*smoothing) / (stats["count"]+smoothing)
            enc_map = stats[te_name].to_dict()
            dev.loc[va_idx_k, te_name] = dev.loc[va_idx_k, col].map(enc_map).fillna(global_mean)
        # holdout: dev 전체 기준으로 인코딩
        stats_full = dev.groupby(col)["y"].agg(["mean", "count"])
        stats_full[te_name] = (stats_full["mean"]*stats_full["count"] + global_mean*smoothing) / (stats_full["count"]+smoothing)
        enc_map_full = stats_full[te_name].to_dict()
        hold[te_name] = hold[col].map(enc_map_full).fillna(global_mean)
        features_track1.append(te_name)
        print(f"  {te_name} 완료")
else:
    print("채택된 TE 변수 없음")

X_t1 = dev[features_track1]
print(f"\nX_t1 shape: {X_t1.shape}  (피처 {len(features_track1)}개)")

In [ ]:
# ==============================================================
# 공통 5-fold CV 러너
# ==============================================================
def run_cv(name, make_model, X_all, y_all, fold_indices,
           n_folds=5, needs_scaling=False, use_sample_weight=False,
           fit_kwargs_fn=None):
    results, oof = [], np.full(len(X_all), -1, dtype=int)
    oof_proba = np.zeros((len(X_all), n_classes))
    last_model = None
    for fold_i, (tr_idx, va_idx) in enumerate(fold_indices[:n_folds]):
        X_tr, X_va = X_all.iloc[tr_idx], X_all.iloc[va_idx]
        y_tr, y_va = y_all.iloc[tr_idx], y_all.iloc[va_idx]
        X_tr, X_va = prepare_X(X_tr, X_va)

        if needs_scaling:
            sc = StandardScaler()
            X_tr = pd.DataFrame(sc.fit_transform(X_tr), columns=X_tr.columns)
            X_va = pd.DataFrame(sc.transform(X_va),  columns=X_va.columns)

        model = make_model()
        kwargs = fit_kwargs_fn(X_va, y_va) if fit_kwargs_fn else {}
        if use_sample_weight:
            kwargs["sample_weight"] = compute_sample_weight("balanced", y_tr)
        model.fit(X_tr, y_tr, **kwargs)

        pred = np.asarray(model.predict(X_va)).ravel().astype(int)
        oof[va_idx] = pred
        if hasattr(model, "predict_proba"):
            oof_proba[va_idx] = model.predict_proba(X_va)
        f1m = f1_score(y_va, pred, average="macro")
        acc = accuracy_score(y_va, pred)
        results.append({"fold": fold_i, "macro_f1": round(f1m,4), "acc": round(acc,4)})
        print(f"  [{name}] fold {fold_i}: Macro-F1={f1m:.4f}, Acc={acc:.4f}")
        last_model = model
    r = pd.DataFrame(results)
    print(f"  [{name}] 평균 Macro-F1 = {r['macro_f1'].mean():.4f} ± {r['macro_f1'].std():.4f}")
    return r, oof, oof_proba, last_model

In [ ]:
# ==============================================================
# ① 로지스틱 회귀 — 전체 dev, GPU(cuML) 우선 / 없으면 CPU 폴백
# ==============================================================
try:
    from cuml.linear_model import LogisticRegression as CuLogisticRegression
    from cuml.preprocessing import StandardScaler as CuStandardScaler
    CUML_AVAILABLE = True
    print("cuML 감지 → GPU 로지스틱 사용")
except ImportError:
    CUML_AVAILABLE = False
    print("cuML 없음 → sklearn CPU 폴백 (solver=saga, n_jobs=-1)")

print("로지스틱 회귀 시작 (전체 dev)...")

if CUML_AVAILABLE:
    results_lr, oof_lr = [], np.full(len(X_t1), -1, dtype=int)
    logreg_proba = np.zeros((len(X_t1), n_classes))
    for fold_i, (tr_idx, va_idx) in enumerate(fold_indices):
        X_tr, X_va = X_t1.iloc[tr_idx], X_t1.iloc[va_idx]
        y_tr, y_va = y_dev.iloc[tr_idx], y_dev.iloc[va_idx]
        X_tr, X_va = prepare_X(X_tr, X_va)
        sc = CuStandardScaler()
        X_tr_s = sc.fit_transform(X_tr.values.astype("float32"))
        X_va_s = sc.transform(X_va.values.astype("float32"))
        m = CuLogisticRegression(max_iter=2000, C=1.0,
                                 class_weight="balanced",
                                 solver="qn", verbose=False)
        m.fit(X_tr_s, y_tr.values.astype("int32"))
        pred = np.asarray(m.predict(X_va_s)).ravel().astype(int)
        oof_lr[va_idx] = pred
        logreg_proba[va_idx] = np.asarray(m.predict_proba(X_va_s))
        f1m = f1_score(y_va, pred, average="macro")
        acc  = accuracy_score(y_va, pred)
        results_lr.append({"fold": fold_i, "macro_f1": round(f1m,4), "acc": round(acc,4)})
        print(f"  [Logistic-GPU] fold {fold_i}: Macro-F1={f1m:.4f}, Acc={acc:.4f}")
    logreg_df  = pd.DataFrame(results_lr)
    logreg_oof = oof_lr
else:
    logreg_df, logreg_oof, logreg_proba, _ = run_cv(
        "Logistic-CPU",
        lambda: LogisticRegression(max_iter=2000, class_weight="balanced",
                                   solver="saga", n_jobs=-1, random_state=SEED),
        X_t1, y_dev, fold_indices,
        needs_scaling=True)

print(f"\n[Logistic] 평균 Macro-F1 = {logreg_df['macro_f1'].mean():.4f} ± {logreg_df['macro_f1'].std():.4f}")

In [ ]:
# ==============================================================
# ② LightGBM — CPU, 5-fold 베이스라인
# ==============================================================
LGB_BASE = dict(objective="multiclass", num_class=n_classes,
                learning_rate=0.05, n_estimators=500,
                max_depth=8, num_leaves=63, min_child_samples=50,
                class_weight="balanced", verbose=-1,
                n_jobs=-1, random_state=SEED)

lgb_df, lgb_oof, lgb_proba, lgb_last = run_cv(
    "LightGBM",
    lambda: lgb.LGBMClassifier(**LGB_BASE),
    X_t1, y_dev, fold_indices,
    fit_kwargs_fn=lambda X_va, y_va: dict(
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)]))

In [ ]:
# ==============================================================
# ③ XGBoost — GPU, 5-fold 베이스라인
# ==============================================================
XGB_BASE = dict(objective="multi:softprob", num_class=n_classes,
                learning_rate=0.05, n_estimators=500,
                max_depth=8, min_child_weight=50,
                subsample=0.8, colsample_bytree=0.8,
                tree_method="hist", device=XGB_DEVICE,
                eval_metric="mlogloss",
                early_stopping_rounds=50,
                random_state=SEED, verbosity=0, n_jobs=-1)

xgb_df, xgb_oof, xgb_proba, xgb_last = run_cv(
    "XGBoost",
    lambda: XGBClassifier(**XGB_BASE),
    X_t1, y_dev, fold_indices,
    use_sample_weight=True,
    fit_kwargs_fn=lambda X_va, y_va: dict(eval_set=[(X_va, y_va)], verbose=False))

In [ ]:
# ==============================================================
# ④ CatBoost — GPU, 5-fold 베이스라인 (튜닝 없음)
# ==============================================================
cat_df, cat_oof, cat_proba, cat_last = run_cv(
    "CatBoost",
    lambda: CatBoostClassifier(loss_function="MultiClass", iterations=500,
                               learning_rate=0.05, depth=8,
                               auto_class_weights="Balanced",
                               task_type=CAT_TASK,
                               random_seed=SEED, verbose=0),
    X_t1, y_dev, fold_indices,
    fit_kwargs_fn=lambda X_va, y_va: dict(eval_set=(X_va, y_va),
                                          early_stopping_rounds=50))

In [ ]:
# ==============================================================
# 4개 모델 비교 표 + 그림
# ==============================================================
comparison = pd.DataFrame({
    "모델": ["Logistic(전체)", "LightGBM(5-fold)", "XGBoost(5-fold,GPU)", "CatBoost(5-fold,GPU)"],
    "Macro_F1": [logreg_df["macro_f1"].mean(), lgb_df["macro_f1"].mean(),
                 xgb_df["macro_f1"].mean(),    cat_df["macro_f1"].mean()],
    "표준편차":  [logreg_df["macro_f1"].std(),  lgb_df["macro_f1"].std(),
                 xgb_df["macro_f1"].std(),     cat_df["macro_f1"].std()],
    "정확도":    [logreg_df["acc"].mean(),       lgb_df["acc"].mean(),
                 xgb_df["acc"].mean(),          cat_df["acc"].mean()],
}).round(4)
print(comparison.to_string(index=False))

plt.figure(figsize=(10, 5))
bars = plt.bar(comparison["모델"], comparison["Macro_F1"],
               yerr=comparison["표준편차"].fillna(0), capsize=5,
               color=["lightcoral","skyblue","lightgreen","gold"])
plt.ylabel("Macro-F1"); plt.title("분류 모델 비교 (트랙1, 베이스라인)")
for b, v in zip(bars, comparison["Macro_F1"]):
    plt.text(b.get_x()+b.get_width()/2, v+0.002, f"{v:.4f}", ha="center", fontsize=9)
plt.xticks(rotation=10); plt.tight_layout()
plt.savefig("../../../figures/20_model_compare.png", dpi=100, bbox_inches="tight")
plt.show()

## Optuna 튜닝
- **LightGBM**: CPU로 튜닝 (n_jobs=-1)
- **XGBoost**: GPU로 튜닝 (device='cuda')
- fold 0으로 빠르게 탐색 → 최적 파라미터로 5-fold 재평가

In [ ]:
# ==============================================================
# Optuna — LightGBM 튜닝 (빠르고 + 크래시 안전)
#   ① 탐색은 층화 서브샘플(20만)로  → 학습 ~8배 빠름 (파라미터 순위는 보존)
#   ② SQLite에 진행 저장 → 죽어도 이어서 재개 (8시간 날리는 일 없음)
#   ③ learning_rate 고정 + early stopping이 n_estimators 결정 → 탐색공간 축소
#   ④ 나쁜 trial은 logloss 보고로 조기 가지치기 → 몇 초 만에 폐기
# ==============================================================
import optuna, json, numpy as np
from sklearn.model_selection import train_test_split

N_TRIALS   = 40                                    # 죽어도 누적됨. 30~50 권장
TUNE_ROWS  = 200_000                               # 탐색용 서브샘플 (150만 → 20만)
STORAGE    = "sqlite:///optuna_lgbm_track1.db"     # ★ 진행이 여기 저장됨
STUDY_NAME = "lgbm_track1_macroF1"

tr_idx, va_idx = fold_indices[0]
X_tr_full, X_va = prepare_X(X_t1.iloc[tr_idx], X_t1.iloc[va_idx])
y_tr_full, y_va = y_dev.iloc[tr_idx].to_numpy(), y_dev.iloc[va_idx].to_numpy()

# 층화 서브샘플: 등급 비율 유지하며 train만 20만 행으로
if len(X_tr_full) > TUNE_ROWS:
    X_tr, _, y_tr, _ = train_test_split(
        X_tr_full, y_tr_full, train_size=TUNE_ROWS,
        stratify=y_tr_full, random_state=SEED)
else:
    X_tr, y_tr = X_tr_full, y_tr_full
print(f"탐색 train={len(X_tr):,}행 / valid={len(X_va):,}행")

def objective(trial):
    params = dict(
        objective="multiclass", num_class=n_classes,
        learning_rate=0.05,                              # ★ 고정
        n_estimators=2000,                               # ★ 크게 + early stopping이 결정
        max_depth=trial.suggest_int("max_depth", 4, 9),
        num_leaves=trial.suggest_int("num_leaves", 31, 127),
        min_child_samples=trial.suggest_int("min_child_samples", 50, 400),
        subsample=trial.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        class_weight="balanced", n_jobs=-1, verbosity=-1,
        force_col_wise=True, random_state=SEED)

    def prune_cb(env):                                   # 나쁜 trial 조기 폐기
        for _, metric, value, _ in env.evaluation_result_list:
            if metric == "multi_logloss":
                trial.report(-value, step=env.iteration)
                if trial.should_prune():
                    raise optuna.TrialPruned()

    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="multi_logloss",
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(0), prune_cb])
    pred = np.asarray(m.predict(X_va)).ravel()
    return f1_score(y_va, pred, average="macro")

study = optuna.create_study(
    study_name=STUDY_NAME, storage=STORAGE, load_if_exists=True,   # ★ 재개
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=100))

done = sum(t.state.name == "COMPLETE" for t in study.trials)
print(f"이미 끝난 trial: {done}개 → 목표 {N_TRIALS}개까지 추가 진행")

study.optimize(objective, n_trials=N_TRIALS,
               timeout=60*60,
               catch=(Exception,),
               show_progress_bar=True)

print(f"\n최적 Macro-F1(fold0, 서브샘플): {study.best_value:.4f}")
best_params = dict(study.best_params, learning_rate=0.05, n_estimators=2000)
print(json.dumps(best_params, indent=2, ensure_ascii=False))

In [ ]:
# ==============================================================
# Optuna — XGBoost 튜닝 (GPU, 5-fold 전체 평균으로 진짜 최적 파라미터 탐색)
# GPU라 trial당 속도가 빠르므로 5-fold도 현실적
# ==============================================================
N_TRIALS_XGB = 50

def xgb_objective_5fold(trial):
    params = dict(
        objective="multi:softprob", num_class=n_classes,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        n_estimators=trial.suggest_int("n_estimators", 300, 2000, step=100),
        max_depth=trial.suggest_int("max_depth", 4, 12),
        min_child_weight=trial.suggest_int("min_child_weight", 10, 300),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        colsample_bylevel=trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        tree_method="hist", device=XGB_DEVICE,
        eval_metric="mlogloss", early_stopping_rounds=30,
        random_state=SEED, verbosity=0, n_jobs=-1)

    scores = []
    for fold_i, (tr_idx, va_idx) in enumerate(fold_indices):
        X_tr, X_va = prepare_X(X_t1.iloc[tr_idx], X_t1.iloc[va_idx])
        y_tr_f = y_dev.iloc[tr_idx]
        y_va_f = y_dev.iloc[va_idx]
        sw = compute_sample_weight("balanced", y_tr_f)
        m = XGBClassifier(**params)
        m.fit(X_tr, y_tr_f, sample_weight=sw,
              eval_set=[(X_va, y_va_f)], verbose=False)
        pred = np.asarray(m.predict(X_va)).ravel().astype(int)
        scores.append(f1_score(y_va_f, pred, average="macro"))

        # Optuna pruning — 처음 2개 fold가 낮으면 조기 중단
        trial.report(np.mean(scores), fold_i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)

xgb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2))
xgb_study.optimize(xgb_objective_5fold, n_trials=N_TRIALS_XGB, show_progress_bar=True)

print(f"\nXGB 최적 Macro-F1(5-fold 평균): {xgb_study.best_value:.4f}")
print(json.dumps(xgb_study.best_params, indent=2))

In [ ]:
# ==============================================================
# LightGBM 튜닝 파라미터로 5-fold 전체 재평가 (CPU)
# ==============================================================
LGB_TUNED = dict(
    objective="multiclass", num_class=n_classes,
    learning_rate=0.05, n_estimators=2000,
    class_weight="balanced",
    verbose=-1, n_jobs=-1, random_state=SEED,
    **study.best_params)

lgb_best_params = LGB_TUNED  # 이후 셀에서 공통 참조

lgb_tuned_df, lgb_tuned_oof, lgb_tuned_proba, lgb_tuned_last = run_cv(
    "LGB-tuned",
    lambda: lgb.LGBMClassifier(**LGB_TUNED),
    X_t1, y_dev, fold_indices,
    fit_kwargs_fn=lambda X_va, y_va: dict(
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)]))
print(f"LGB 튜닝 전 {lgb_df['macro_f1'].mean():.4f} → 후 {lgb_tuned_df['macro_f1'].mean():.4f}")

In [ ]:
# ==============================================================
# XGBoost 튜닝 파라미터로 5-fold 재확인 (GPU)
# xgb_study 자체가 이미 5-fold 기준이므로 이 셀은 최종 OOF 확률 수집용
# ==============================================================
xgb_best_params = dict(
    objective="multi:softprob", num_class=n_classes,
    tree_method="hist", device=XGB_DEVICE,
    eval_metric="mlogloss", early_stopping_rounds=30,
    random_state=SEED, verbosity=0, n_jobs=-1,
    **xgb_study.best_params)

xgb_tuned_df, xgb_tuned_oof, xgb_tuned_proba, xgb_tuned_last = run_cv(
    "XGB-tuned",
    lambda: XGBClassifier(**xgb_best_params),
    X_t1, y_dev, fold_indices,
    use_sample_weight=True,
    fit_kwargs_fn=lambda X_va, y_va: dict(eval_set=[(X_va, y_va)], verbose=False))
print(f"XGB 튜닝 전 {xgb_df['macro_f1'].mean():.4f} → 후 {xgb_tuned_df['macro_f1'].mean():.4f}")
print(f"(XGB 튜닝은 5-fold 평균 기준으로 탐색됨)")

## 스태킹 앙상블 — LightGBM(튜닝) + XGBoost(튜닝)
OOF 확률값을 메타 피처로 사용해 Logistic Regression 메타 모델 학습

In [ ]:
# ==============================================================
# 스태킹 앙상블 — OOF 확률 → 메타 모델
# ==============================================================
# 메타 피처: LGB 튜닝 OOF 확률(16) + XGB 튜닝 OOF 확률(16) = 32차원
meta_train = np.hstack([lgb_tuned_proba, xgb_tuned_proba])
print(f"메타 피처 shape: {meta_train.shape}")

# 메타 모델 — LogisticRegression (OOF 위에서 학습하므로 과적합 없음)
meta_model = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced",
                                solver="lbfgs", multi_class="multinomial",
                                n_jobs=-1, random_state=SEED)
meta_model.fit(meta_train, y_dev)

stack_oof_pred = meta_model.predict(meta_train)
stack_f1 = f1_score(y_dev, stack_oof_pred, average="macro")
stack_acc = accuracy_score(y_dev, stack_oof_pred)
print(f"\n[스태킹] OOF Macro-F1: {stack_f1:.4f}  |  정확도: {stack_acc:.4f}")
print(f"  LGB-tuned  OOF: {lgb_tuned_df['macro_f1'].mean():.4f}")
print(f"  XGB-tuned  OOF: {xgb_tuned_df['macro_f1'].mean():.4f}")
print(f"  스태킹  OOF:    {stack_f1:.4f}")

In [ ]:
# ==============================================================
# 혼동행렬 + 클래스별 리포트 (스태킹 OOF 기준)
# ==============================================================
labels = GRADE_ORDER
cm = confusion_matrix(y_dev, stack_oof_pred, labels=range(n_classes))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title("혼동행렬 (마리수, 스태킹 OOF)"); axes[0].set_xlabel("예측"); axes[0].set_ylabel("실제")
sns.heatmap(cm_norm, annot=True, fmt=".0f", cmap="YlOrRd",
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title("혼동행렬 (행 기준 %)"); axes[1].set_xlabel("예측"); axes[1].set_ylabel("실제")
plt.tight_layout()
plt.savefig("../../../figures/20_confusion_stacking.png", dpi=100, bbox_inches="tight")
plt.show()

print(classification_report(y_dev, stack_oof_pred,
                            labels=range(n_classes),
                            target_names=labels, zero_division=0))

In [ ]:
# ==============================================================
# SHAP 변수 중요도 — LGB 튜닝 모델 기준 (shap 0.45+ 3차원 대응)
# ==============================================================
tr_idx_last, va_idx_last = fold_indices[-1]
X_sample = prepare_X(X_t1.iloc[tr_idx_last], X_t1.iloc[va_idx_last])[1] \
               .sample(min(1000, len(va_idx_last)), random_state=SEED)

explainer = shap.TreeExplainer(lgb_tuned_last)
sv = explainer.shap_values(X_sample)

if isinstance(sv, list):
    mean_abs = np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)
    get_class_sv = lambda k: sv[k]
else:
    print(f"SHAP 3차원 배열: {sv.shape}")
    mean_abs = np.abs(sv).mean(axis=(0, 2))
    get_class_sv = lambda k: sv[:, :, k]

imp = (pd.DataFrame({"변수": X_sample.columns, "SHAP중요도": mean_abs})
         .sort_values("SHAP중요도", ascending=False))
print(imp.head(20).to_string(index=False))

plt.figure(figsize=(10, 8))
top20 = imp.head(20).iloc[::-1]
plt.barh(top20["변수"], top20["SHAP중요도"], color="steelblue")
plt.xlabel("평균 |SHAP|"); plt.title("SHAP 변수 중요도 (LGB-tuned, 상위 20)")
plt.tight_layout()
plt.savefig("../../../figures/20_shap_importance.png", dpi=100, bbox_inches="tight")
plt.show()

k = grade_to_idx["1++A"]
shap.summary_plot(get_class_sv(k), X_sample, max_display=15, show=False)
plt.title("SHAP Summary — 1++A 예측 (LGB-tuned)")
plt.tight_layout()
plt.savefig("../../../figures/20_shap_summary_1ppA.png", dpi=100, bbox_inches="tight")
plt.show()

imp.to_csv("../../../data/processed/4_model/20_shap_importance.csv",
           index=False, encoding="utf-8-sig")

In [ ]:
# ==============================================================
# 홀드아웃 최종 평가 — 딱 한 번만!
# 스태킹: dev 전체로 LGB+XGB 재학습 → 홀드아웃 확률 → 메타 모델 예측
# ==============================================================
X_dev_full, X_hold_prep = prepare_X(dev[features_track1], hold[features_track1])
y_hold = hold["y"]

# LGB 홀드아웃 확률
hold_lgb = lgb.LGBMClassifier(**lgb_best_params)
hold_lgb.fit(X_dev_full, y_dev)
p_hold_lgb = hold_lgb.predict_proba(X_hold_prep)

# XGB 홀드아웃 확률 (GPU)
sw_dev = compute_sample_weight("balanced", y_dev)
hold_xgb = XGBClassifier(**xgb_best_params)
hold_xgb.fit(X_dev_full, y_dev, sample_weight=sw_dev)
p_hold_xgb = hold_xgb.predict_proba(X_hold_prep)

# 스태킹 예측
meta_hold = np.hstack([p_hold_lgb, p_hold_xgb])
pred_hold_stack = meta_model.predict(meta_hold)

hold_f1  = f1_score(y_hold, pred_hold_stack, average="macro")
hold_acc = accuracy_score(y_hold, pred_hold_stack)
print(f"홀드아웃 스태킹 Macro-F1: {hold_f1:.4f}")
print(f"홀드아웃 스태킹 정확도:    {hold_acc:.4f}")
print(f"(참고) CV LGB-tuned: {lgb_tuned_df['macro_f1'].mean():.4f} / XGB-tuned: {xgb_tuned_df['macro_f1'].mean():.4f}")

In [ ]:
# ==============================================================
# 결과 저장 (지표 + 파라미터)
# ==============================================================
summary = {
    "logistic_cv":       round(logreg_df["macro_f1"].mean(), 4),
    "lgb_cv":            round(lgb_df["macro_f1"].mean(), 4),
    "xgb_cv":            round(xgb_df["macro_f1"].mean(), 4),
    "cat_cv":            round(cat_df["macro_f1"].mean(), 4),
    "lgb_tuned_cv":      round(lgb_tuned_df["macro_f1"].mean(), 4),
    "xgb_tuned_cv":      round(xgb_tuned_df["macro_f1"].mean(), 4),
    "stacking_oof_f1":   round(stack_f1, 4),
    "holdout_stack_f1":  round(hold_f1, 4),
    "holdout_stack_acc": round(hold_acc, 4),
    "lgb_best_params":   study.best_params,
    "xgb_best_params":   xgb_study.best_params,
    "n_features":        len(features_track1),
}
with open("../../../data/processed/4_model/20_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 대회 test 예측 & 제출 파일 생성
dev + holdout 전체(라벨 100%)로 최종 모델 재학습 → test 예측 → 제출 파일

In [ ]:
# ==============================================================
# test_hanwoo.csv → 피처 빌드 (train과 동일 파이프라인)
# ==============================================================
def build_features_for_test(test_path="../../../data/test/test_hanwoo.csv"):
    test = pd.read_csv(test_path, low_memory=False)
    for c in ["ABATT_DATE", "BIRTH_YMD", "JUDGE_DATE"]:
        test[c] = pd.to_datetime(test[c], errors="coerce")

    death = pd.read_csv("../../../data/raw/hanwoo_death.csv")
    test = test.merge(death.groupby("FARM_UNIQUE_NO").size()
                      .rename("death_count").reset_index(),
                      on="FARM_UNIQUE_NO", how="left")
    area = pd.read_csv("../../../data/raw/hanwoo_area.csv")
    for c in ["C2023", "C2024", "C2025", "AREA"]:
        area[c] = area[c].replace(-99, np.nan)
    test = test.merge(area.groupby("FARM_UNIQUE_NO")[["C2023","C2024","C2025","AREA"]]
                      .sum().reset_index(), on="FARM_UNIQUE_NO", how="left")
    lin = pd.read_csv("../../../data/raw/hanwoo_lineage_0612.csv", low_memory=False)
    test = test.merge(lin, on="CATTLE_NO", how="left")

    w = pd.read_csv("../../../data/processed/1_merge/step4-3_weather.csv",
                    usecols=["stn","date","THI_grade","rn_day","ws_davg","ta_min"],
                    low_memory=False)
    w["date"] = pd.to_datetime(w["date"])
    for g in ["양호","주의","경고","위험","폐사"]:
        w[f"d_{g}"] = (w["THI_grade"] == g).astype(int)
    sum_cols = ["d_양호","d_주의","d_경고","d_위험","d_폐사","rn_day","ws_davg","ta_min"]
    test = test.reset_index(drop=True)
    n = len(test)
    days_arr = np.full(n, np.nan); seg = np.full((n, len(sum_cols)), np.nan)
    for stn, ws in w.groupby("stn"):
        ws = ws.sort_values("date"); dates = ws["date"].values
        csum = np.vstack([np.zeros(len(sum_cols)), ws[sum_cols].cumsum().values])
        idx = np.where(test["stn"].values == stn)[0]
        if len(idx) == 0: continue
        lo = np.searchsorted(dates, test["BIRTH_YMD"].values[idx], side="left")
        hi = np.searchsorted(dates, test["ABATT_DATE"].values[idx], side="right")
        seg[idx] = csum[hi] - csum[lo]; days_arr[idx] = hi - lo
    test["days_total"] = days_arr
    for i, g in enumerate(["양호","주의","경고","위험","폐사"]):
        test[f"days_{g}"] = seg[:, i]
    with np.errstate(invalid="ignore", divide="ignore"):
        test["rn_day_mean"]  = seg[:, 5] / days_arr
        test["ws_davg_mean"] = seg[:, 6] / days_arr
        test["ta_min_mean"]  = seg[:, 7] / days_arr

    test["ratio_고온"]   = (test["days_주의"]+test["days_경고"]+test["days_위험"]) / test["days_total"]
    test["ratio_강더위"] = (test["days_경고"]+test["days_위험"]) / test["days_total"]
    test["ratio_위험"]   = test["days_위험"] / test["days_total"]

    test["abatt_year"]    = test["ABATT_DATE"].dt.year
    test["abatt_month"]   = test["ABATT_DATE"].dt.month
    test["abatt_quarter"] = test["ABATT_DATE"].dt.quarter
    test["birth_year"]    = test["BIRTH_YMD"].dt.year
    test["birth_month"]   = test["BIRTH_YMD"].dt.month
    def to_season(m):
        return ("봄" if m in [3,4,5] else "여름" if m in [6,7,8]
                else "가을" if m in [9,10,11] else "겨울")
    test["abatt_season"] = test["abatt_month"].map(to_season)
    test["birth_season"] = test["birth_month"].map(to_season)
    test["fattening_days"] = (test["ABATT_DATE"] - test["BIRTH_YMD"]).dt.days
    test["daily_gain"]     = test["WEIGHT"] / test["fattening_days"]
    test["Age_squared"]    = test["AGE"] ** 2
    test["age_optimal"]    = ((test["AGE"] >= 28) & (test["AGE"] <= 32)).astype(int)
    test["heat_high"]      = (test["ratio_고온"] >= HEAT_Q75).astype(int)
    test["density"]        = (test["C2025"] / test["AREA"]).replace([np.inf,-np.inf], np.nan)
    test["death_rate"]     = (test["death_count"] / test["C2025"]).replace([np.inf,-np.inf], np.nan)
    test["density_x_heat"] = test["density"] * test["ratio_고온"]

    for col in ["KPN_NO","FATHER_CATTLE_NO","MOTHER_ANIMAL_NO"]:
        freq_map = (df[[col, f"{col}_freq"]].dropna()
                      .drop_duplicates().set_index(col)[f"{col}_freq"].to_dict())
        test[f"{col}_freq"] = test[col].map(freq_map)

    g_ebv = pd.read_excel("../../../data/raw/KPN 유전능력 자료.xlsx")
    g_ebv["KPN명호"] = g_ebv["KPN명호"].astype(str).str.strip()
    g_ebv = g_ebv[~g_ebv["KPN명호"].isin(["nan",""])].drop_duplicates("KPN명호")
    emap = {"근내지방도 육종가":"ebv_marbling","근내지방도 정확도":"ebv_marbling_acc",
            "도체중 육종가":"ebv_cwt","등심단면적 육종가":"ebv_rea",
            "등지방두께 육종가":"ebv_backfat","12개월체중 육종가":"ebv_wt12"}
    gg = g_ebv[["KPN명호"]+list(emap)].rename(columns={"KPN명호":"KPN_NO",**emap})
    for c in emap.values():
        gg[c] = pd.to_numeric(gg[c], errors="coerce")
    test["KPN_NO"] = test["KPN_NO"].astype(str).str.strip()
    test = test.merge(gg, on="KPN_NO", how="left")
    test["has_ebv"] = test["ebv_marbling"].notna().astype(int)

    rec = df.dropna(subset=["CATTLE_NO"]).drop_duplicates("CATTLE_NO").set_index("CATTLE_NO")
    rec_jd = pd.to_datetime(rec["JUDGE_DATE"], errors="coerce")
    rec_g  = pd.to_numeric(rec["grade_num"], errors="coerce")
    for idcol, nm in [("MOTHER_ANIMAL_NO","mother"),("FATHER_CATTLE_NO","father")]:
        a_jd = test[idcol].map(rec_jd); a_g = test[idcol].map(rec_g)
        gated = np.where(a_jd.values < test["JUDGE_DATE"].values, a_g.values, np.nan)
        test[f"{nm}_own_grade"] = gated
        test[f"has_{nm}_grade"] = (~np.isnan(gated)).astype(int)

    peer_w = df.groupby(["AGE","JUDGE_SEX"])["WEIGHT"].mean().rename("peer_w")
    test = test.merge(peer_w.reset_index(), on=["AGE","JUDGE_SEX"], how="left")
    test["weight_vs_peer"] = test["WEIGHT"] / test["peer_w"]
    test = test.drop(columns=["peer_w"])

    test["sex"] = test["JUDGE_SEX"]
    dummies = pd.get_dummies(test[["sido","abatt_season","birth_season","sex"]],
                             prefix=["sido","abatt_season","birth_season","sex"]).astype(int)
    train_dummy_cols = [c for c in features_track1
                        if c.startswith(("sido_","abatt_season_","birth_season_","sex_"))]
    for col in train_dummy_cols:
        test[col] = dummies[col] if col in dummies.columns else 0

    # ── 채택된 단순 파생변수 (dev 통계 기준, 누수 없음) ──
    test["daily_gain_squared"] = test["daily_gain"] ** 2
    age_mean = dev.groupby("AGE")["WEIGHT"].mean()
    test["weight_vs_age_peer"] = test["WEIGHT"] / test["AGE"].map(age_mean)
    test["lineage_missing"] = test["KPN_NO_freq"].isnull().astype(int)
    q75_gain = dev["daily_gain"].quantile(0.75)
    test["gain_high"] = (test["daily_gain"] >= q75_gain).astype(int)

    # ── 채택된 OOF TE: dev 전체 기준 맵으로 test에 적용 ──
    for te_name in adopted_te:
        col, smoothing = TE_CANDIDATES[te_name]
        global_mean = dev["y"].mean()
        stats_full = dev.groupby(col)["y"].agg(["mean", "count"])
        stats_full[te_name] = (stats_full["mean"]*stats_full["count"] + global_mean*smoothing) / (stats_full["count"]+smoothing)
        enc_map_full = stats_full[te_name].to_dict()
        test[te_name] = test[col].map(enc_map_full).fillna(global_mean)

    return test

test_feat = build_features_for_test()
print(f"test 피처화 완료: {test_feat.shape}")
missing_cols = [c for c in features_track1 if c not in test_feat.columns]
assert not missing_cols, f"누락 피처: {missing_cols}"
print(f"트랙1 {len(features_track1)}개 피처 전부 생성 확인")

In [ ]:
# ==============================================================
# 최종 모델 — dev+holdout 전체로 재학습 → test 스태킹 예측 → 제출 파일
# ==============================================================
train_all = pd.concat([dev, hold], ignore_index=True)
y_all = train_all["y"]

X_all_full = train_all[features_track1].replace([np.inf,-np.inf], np.nan)
train_median = X_all_full.median(numeric_only=True)
X_all_full = X_all_full.fillna(train_median)

X_test = (test_feat[features_track1].replace([np.inf,-np.inf], np.nan)
                                    .fillna(train_median))
drop_bad = X_all_full.columns[X_all_full.isnull().any()].tolist()
if drop_bad:
    X_all_full = X_all_full.drop(columns=drop_bad)
    X_test     = X_test.drop(columns=drop_bad)

sw_all = compute_sample_weight("balanced", y_all)

# LGB 최종
final_lgb = lgb.LGBMClassifier(**lgb_best_params)
final_lgb.fit(X_all_full, y_all)
p_test_lgb = final_lgb.predict_proba(X_test)

# XGB 최종 (GPU)
final_xgb = XGBClassifier(**xgb_best_params)
final_xgb.fit(X_all_full, y_all, sample_weight=sw_all)
p_test_xgb = final_xgb.predict_proba(X_test)

# 스태킹 최종 예측
meta_test = np.hstack([p_test_lgb, p_test_xgb])
pred_idx   = meta_model.predict(meta_test)
pred_grade = pd.Series(pred_idx).map(idx_to_grade)

submission = pd.DataFrame({
    "CATTLE_NO":  test_feat["CATTLE_NO"],
    "LAST_GRADE": pred_grade.values,
})
print(submission["LAST_GRADE"].value_counts().reindex(GRADE_ORDER).to_string())
assert submission["LAST_GRADE"].isnull().sum() == 0, "예측 결측 존재!"
assert len(submission) == 452497, "행 수 불일치!"

submission.to_csv("../../../data/processed/4_model/20_submission.csv",
                  index=False, encoding="utf-8-sig")
joblib.dump(final_lgb, "../../../models/20_lgb_final.pkl")
joblib.dump(final_xgb, "../../../models/20_xgb_final.pkl")
joblib.dump(meta_model, "../../../models/20_meta_model.pkl")
print("\n제출 파일: 20_submission.csv / 모델: 20_lgb_final.pkl, 20_xgb_final.pkl, 20_meta_model.pkl")